### Multi-Agent Bidding Negotiation Training Pipeline

**Project:** Enterprise Support Router

This Colab notebook demonstrates the TRL GRPO + Unsloth training pipeline for our 4-Agent Negotiation Protocol. We leverage **Unsloth** for 4-bit Llama-3.2-1B inference, and **TRL** to optimize 4 distinct agent personas collaboratively.

In [ ]:
# 1. Install Requirements
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install "openenv-core[core]>=0.2.1"

In [ ]:
# 0. Verify GPU is available
import torch
print(f"✅ GPU Available: {torch.cuda.is_available()}")
print(f"✅ GPU Name: {torch.cuda.get_device_name(0)}")
print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


### 2. Load the Unsloth 4-bit LLM Backbone

In [ ]:
import torch
from unsloth import FastLanguageModel

print("Loading Unsloth optimized Llama 3.2 1B Instruct model in 4-bit mode...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)
model = FastLanguageModel.for_training(model)
print("Model loaded successfully!")

### 3. Initialize the Multi-Agent GRPO Pipeline
Because our OpenEnv requires a strict 3-Phase State Machine, we generate structured synthetic JSON trajectories for the Multi-Agent Personas to optimize against.

In [ ]:
import os
import sys

# Clone the entire project repository to ensure all local imports work
!git clone https://github.com/RavichandraNayakar/openenv-hackathon-project.git
%cd openenv-hackathon-project
sys.path.append(os.getcwd())

print("Cloned repository and set up paths for training.")


### 4. Run the Training Protocol
We train the 4 underlying Agent Personas sequentially using GRPO.

In [ ]:
from my_env.pytorch.training.trl_multi_agent_trainer import MultiAgentGRPOTrainer

# DRY RUN CONFIGURATION
trainer = MultiAgentGRPOTrainer(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct",
    learning_rate=2e-5,
    batch_size=2,
    gradient_accumulation_steps=4,      
    num_train_epochs=2,   
)
print("Initializing GRPO alignment...")

# Kick off the dry run
trainer.train_all_agents(output_dir="./checkpoints_multi_agent")


## Results Verification

In [ ]:
# 5. Verify Training Output
import os, json

history_path = "./checkpoints_multi_agent/training_history.json"
if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    print("✅ Training Complete! Agent Performance Summary:")
    print("="*50)
    for agent, data in history.items():
        rate = data["success_rate"][-1] if data["success_rate"] else 0
        loss = data["runs"][-1]["final_loss"] if data["runs"] else "N/A"
        print(f"  {agent:<12} | Success: {rate:.1%} | Final Loss: {loss}")
    print("="*50)
else:
    print("❌ training_history.json not found. Training may have failed.")
